In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

train_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/train_engineered.csv"
)

val_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/validation_engineered.csv"
)

feature_columns = (
    PROJECT_ROOT / "docs/feature-columns.txt"
).read_text(encoding="utf-8-sig").splitlines()

X_train = train_df[feature_columns]
y_train = train_df["RUL"]

X_val = val_df[feature_columns]
y_val = val_df["RUL"]

assert set(train_df.engine_id).isdisjoint(set(val_df.engine_id))
assert len(feature_columns) == 123

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Data ready for training!")


Training: (16561, 123)
Validation: (4070, 123)
Data ready for training!


In [ ]:

models = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Ridge Regression": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = np.maximum(model.predict(X_val), 0)

    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse
    })

    print(f"{name}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}\n")

results_df = pd.DataFrame(results)

display(results_df)

results_df.to_csv(
    PROJECT_ROOT / "ml/reports/baseline_results.csv",
    index=False
)
